# 中证800 V60：V46 离线组合回测器与真实收益重算

## 实验目的

这个 notebook 不是继续发明新模型，而是修正研究评估链路：让 notebook 的离线评估尽量接近 `jq_backtest_v46_legacy_unsealed_monitor.py` 的真实月度调仓逻辑，并在聚宽研究环境中对选中 target 用 `get_price` 重算 close-to-close 真实收益。

核心目标：

1. 固定 V46/V56 的 direct LGB 训练 recipe：`alpha_1m + fixed 120 + legacy_unsealed_q4`。
2. 支持多个训练起止日期和 feature variant。
3. 不只看 IC 或 label topN，而是生成真实使用方式下的月度组合：`score top30 -> industry-neutral top6 -> equal weight`。
4. 输出 proxy 口径与 JoinQuant close-to-close realized 口径的 gross/net return、excess、turnover、drawdown、target list、同约束随机分位。
5. 只把通过离线组合回测的模型导出为 pkl，再进入 JoinQuant 回测验收。

重要边界：离线 CSV 如果没有停牌、ST、涨跌停、当日价格等字段，本 notebook 会明确标记无法离线复刻这些过滤，不会假装和 JoinQuant 回测完全等价。

V60 相比 V59 的唯一核心变化：新增 selected-target realized return rebuild block。它只对每月已经选出来的 target 拉行情，数据量很小，用于检查模型排序是否更接近真实 JoinQuant 回测。

In [ ]:
import os
import gc
import pickle
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", every=None):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc)
    if every is None:
        every = max(1, int((total or 100) / 20))

    def _gen():
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                if total is None:
                    print("%s %s" % (desc, i))
                else:
                    print("%s %s/%s" % (desc, i, total))
            yield item
    return _gen()


# =========================
# Config
# =========================
DATA_PATH = "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"
OUT_DIR = "csi800_ml_v60_realized_return_rebuild_outputs"

TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
INDUSTRY_COL = "industry_bucket"
BENCHMARK = "000906.XSHG"
MODEL_FAMILY = "v60_v46_offline_portfolio_realized_rebuild"
BOUNDARY_POLICY = "legacy_unsealed_q4"
USE_LEGACY_UNSEALED_BOUNDARY = True

# 灵活控制训练窗口；只改这里即可增加/删除模型银行成员。
TRAIN_WINDOW_SPECS = [
    {"tag": "2019_2023", "train_start": "2019-01-01", "train_end": "2023-12-31", "test_start": "2024-01-01"},
    {"tag": "2019_2024", "train_start": "2019-01-01", "train_end": "2024-12-31", "test_start": "2025-01-01"},
    {"tag": "2019_2025", "train_start": "2019-01-01", "train_end": "2025-12-31", "test_start": "2026-01-01"},
]

FIXED_ITER = 120
SEED = 42
CORR_THRESHOLD = 0.70
TOP_N_CANDIDATES = 30
STOCK_NUM = 6
INDUSTRY_CAP_RATIO = 0.20

# 近似 JoinQuant 回测成本：买卖均有滑点和佣金，卖出额外印花税。
# 这是月度组合模拟，不是逐笔成交模拟；最终仍以 JoinQuant 回测验收。
SLIPPAGE_RATE = 0.00246
OPEN_COMMISSION = 0.0003
CLOSE_COMMISSION = 0.0003
CLOSE_TAX = 0.001
BUY_COST_RATE = SLIPPAGE_RATE + OPEN_COMMISSION
SELL_COST_RATE = SLIPPAGE_RATE + CLOSE_COMMISSION + CLOSE_TAX

# 离线过滤只在 CSV 有对应字段时启用；缺字段会在 filter_report 里说明。
USE_OFFLINE_TRADABILITY_FILTER = True
MIN_LISTING_DAYS = 180
INITIAL_CASH_FOR_MIN_LOT = 500000.0
KCB_MIN_LOT = 200
NORMAL_MIN_LOT = 100

RANDOM_SIM_N = 2000
RANDOM_SEED = 42
REALIZED_TOP_N = 6
REALIZED_TOP20_N = 20
EXPORT_MODELS = True

# AUTO: 有 JoinQuant get_price 时自动重算真实 close-to-close 收益；本地普通 Python 环境自动跳过。
RUN_JQ_REALIZED_REBUILD = "AUTO"
REALIZED_PRICE_FIELD = "close"
REALIZED_PRICE_FQ = "pre"
REALIZED_PRICE_CHUNK_SIZE = 120

os.makedirs(OUT_DIR, exist_ok=True)
print("output dir:", OUT_DIR)
print("data path:", DATA_PATH)


## 特征分组

`full` 对齐 V46/V56 的 hybrid-light feature stack。`light_technical_price` 保留较粗的月频动量/风险特征，去掉更细的量价/技术/时序价格特征。`no_technical_price` 是负向对照，不作为当前候选优先级。

In [ ]:
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio",
    "book_to_price_ratio",
    "earnings_yield",
    "sales_to_price_ratio",
    "cash_earnings_to_price_ratio",
    "earnings_to_price_ratio",
    "roe_ttm",
    "roa_ttm",
    "gross_profit_ttm",
    "operating_profit_to_total_profit",
    "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage",
    "adjusted_profit_to_total_profit",
    "ACCA",
    "growth",
    "net_working_capital",
    "operating_profit_per_share",
    "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share",
    "super_quick_ratio",
    "MLEV",
    "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio",
    "momentum",
    "Rank1M",
    "sharpe_ratio_60",
    "Variance20",
    "liquidity",
    "beta",
    "ATR6",
    "MFI14",
    "DAVOL10",
    "VOL10",
    "VMACD",
    "VOSC",
    "Skewness20",
    "Kurtosis20",
]

HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60",
    "liq_paused_count_20",
    "px_close_to_ma60",
    "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m",
    "ts_Rank1M_rank_chg_1m",
]

FULL_FEATURE_COLS = BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS

LIGHT_TECHNICAL_KEEP = [
    "momentum",
    "Rank1M",
    "sharpe_ratio_60",
    "Variance20",
    "beta",
]

TECHNICAL_PRICE_COLS = [
    "momentum",
    "Rank1M",
    "sharpe_ratio_60",
    "Variance20",
    "beta",
    "ATR6",
    "MFI14",
    "DAVOL10",
    "VOL10",
    "VMACD",
    "VOSC",
    "Skewness20",
    "Kurtosis20",
    "liq_money_ratio_20_60",
    "liq_paused_count_20",
    "px_close_to_ma60",
    "px_drawdown_60",
    "ts_Rank1M_rank_chg_1m",
]

LIGHT_TECHNICAL_REMOVE = [c for c in TECHNICAL_PRICE_COLS if c not in LIGHT_TECHNICAL_KEEP]
NO_TECHNICAL_REMOVE = list(TECHNICAL_PRICE_COLS)

FEATURE_VARIANTS = [
    {
        "feature_variant": "full",
        "description": "V46/V56 full hybrid-light feature stack",
        "candidate_cols": list(FULL_FEATURE_COLS),
    },
    {
        "feature_variant": "light_technical_price",
        "description": "keep coarse monthly momentum/risk style; remove fine price-volume/technical extras",
        "candidate_cols": [c for c in FULL_FEATURE_COLS if c not in LIGHT_TECHNICAL_REMOVE],
    },
    {
        "feature_variant": "no_technical_price",
        "description": "remove technical/price/momentum-risk family; keep fundamental/liquidity style and cashflow temporal feature",
        "candidate_cols": [c for c in FULL_FEATURE_COLS if c not in NO_TECHNICAL_REMOVE],
    },
]

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}

variant_manifest_rows = []
for v in FEATURE_VARIANTS:
    cols = list(v["candidate_cols"])
    variant_manifest_rows.append({
        "feature_variant": v["feature_variant"],
        "candidate_feature_count": len(cols),
        "description": v["description"],
        "candidate_features": ",".join(cols),
        "removed_from_full": ",".join([c for c in FULL_FEATURE_COLS if c not in cols]),
    })
variant_manifest_df = pd.DataFrame(variant_manifest_rows)
display(variant_manifest_df[["feature_variant", "candidate_feature_count", "description", "removed_from_full"]])


## 基础 helper

这里都是研究环境通用函数：安全日期转换、RankIC、回撤、相关性去冗余、训练矩阵准备。保持简单 pandas 写法，避免旧环境兼容问题。

In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def safe_to_datetime(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col])
    return out


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def calc_nav(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if len(s) == 0:
        return pd.Series(dtype=float)
    return (1.0 + s).cumprod()


def calc_drawdown_from_returns(ret_series):
    nav = calc_nav(ret_series)
    if len(nav) == 0:
        return np.nan
    dd = nav / nav.cummax() - 1.0
    return float(dd.min())


def summarize_return_series(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return {"months": 0, "cum_ret": np.nan, "mean_ret": np.nan, "win_rate": np.nan, "max_drawdown": np.nan}
    return {
        "months": int(len(s)),
        "cum_ret": float((1.0 + s).prod() - 1.0),
        "mean_ret": float(s.mean()),
        "win_rate": float((s > 0).mean()),
        "max_drawdown": calc_drawdown_from_returns(s),
    }


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    comps = []

    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)

    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    if len(cols) == 0:
        raise ValueError("no candidate feature exists in train data")
    missing = train_df[cols].isnull().sum().to_dict()
    keep = []
    remove = []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            comp = sorted(comp, key=lambda x: (missing[x], x))
            keep.append(comp[0])
            remove.extend(comp[1:])
    return keep, remove


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    y = d[target_col].astype(float).copy()
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, y, fill_values, d.index


def split_diag_valid(train_df):
    months = sorted(pd.to_datetime(train_df[DATE_COL].dropna().unique()))
    if len(months) <= 8:
        return train_df.copy(), train_df.copy()
    n_valid = max(6, int(round(len(months) * 0.20)))
    valid_months = set(months[-min(n_valid, len(months) - 1):])
    fit = train_df[~train_df[DATE_COL].isin(valid_months)].copy()
    valid = train_df[train_df[DATE_COL].isin(valid_months)].copy()
    if fit.empty or valid.empty:
        return train_df.copy(), train_df.copy()
    return fit, valid


## 数据加载与收益口径

默认读取固定 CSV。收益口径按优先级自动识别：

1. `raw_return_1m` 或 `stock_return_1m` 作为组合 raw return。
2. `benchmark_csi800_1m` 或 `benchmark_alla_1m` 作为 benchmark return。
3. `alpha_1m` 作为 excess label。

如果 raw/benchmark 不存在，会退化到 `alpha_1m`，并在输出中标记。

In [ ]:
def first_existing(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None


def load_dataset(path):
    if not os.path.exists(path):
        raise IOError("data csv not found: " + path)
    df = pd.read_csv(path)
    df = safe_to_datetime(df, ["rebalance_date", "feature_date", "next_date"])
    if STOCK_COL not in df.columns:
        for alt in ["code", "security", "order_book_id"]:
            if alt in df.columns:
                df = df.rename(columns={alt: STOCK_COL})
                break
    if TARGET_COL not in df.columns:
        if "raw_return_1m" in df.columns and "benchmark_csi800_1m" in df.columns:
            df[TARGET_COL] = df["raw_return_1m"] - df["benchmark_csi800_1m"]
        else:
            raise ValueError("target column not found: " + TARGET_COL)
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    need = [STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, "feature_date", "next_date"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError("dataset missing columns: " + ",".join(missing))
    return df


def infer_return_columns(df):
    raw_col = first_existing(df.columns, ["raw_return_1m", "stock_return_1m", "return_1m", "next_return_1m"])
    bench_col = first_existing(df.columns, ["benchmark_csi800_1m", "benchmark_000906_1m", "benchmark_alla_1m", "cum_csi800_1m"])
    alpha_col = TARGET_COL if TARGET_COL in df.columns else None
    if raw_col is None and alpha_col is not None and bench_col is not None:
        df["_v59_raw_return_1m"] = df[alpha_col] + df[bench_col]
        raw_col = "_v59_raw_return_1m"
    return raw_col, bench_col, alpha_col


def normalize_train_spec(spec):
    out = dict(spec)
    out["train_start"] = pd.Timestamp(out.get("train_start", "2019-01-01"))
    out["train_end"] = pd.Timestamp(out["train_end"])
    out["test_start"] = pd.Timestamp(out.get("test_start", out["train_end"] + pd.Timedelta(days=1)))
    if not out.get("tag"):
        out["tag"] = "{}_{}".format(out["train_start"].strftime("%Y%m%d"), out["train_end"].strftime("%Y%m%d"))
    return out


def make_train_df(df_all, spec):
    spec = normalize_train_spec(spec)
    mask = (df_all[DATE_COL] >= spec["train_start"]) & (df_all[DATE_COL] <= spec["train_end"])
    if not USE_LEGACY_UNSEALED_BOUNDARY:
        mask = mask & (df_all["next_date"] <= spec["train_end"])
    return df_all[mask].copy()


def make_test_df(df_all, spec):
    spec = normalize_train_spec(spec)
    return df_all[df_all[DATE_COL] >= spec["test_start"]].copy()


df_all = load_dataset(DATA_PATH)
RAW_RET_COL, BENCH_RET_COL, ALPHA_RET_COL = infer_return_columns(df_all)
print("loaded:", df_all.shape)
print(df_all[["rebalance_date", "feature_date", "next_date"]].agg(["min", "max"]))
print("target:", TARGET_COL)
print("raw return col:", RAW_RET_COL)
print("benchmark return col:", BENCH_RET_COL)
print("alpha return col:", ALPHA_RET_COL)
display(df_all[[TARGET_COL]].describe())

available_rows = []
for v in FEATURE_VARIANTS:
    cols = [c for c in v["candidate_cols"] if c in df_all.columns]
    available_rows.append({
        "feature_variant": v["feature_variant"],
        "configured": len(v["candidate_cols"]),
        "available": len(cols),
        "missing": ",".join([c for c in v["candidate_cols"] if c not in df_all.columns]),
    })
available_df = pd.DataFrame(available_rows)
display(available_df)


## 离线可交易过滤

这一步只在 CSV 有字段时生效。它的作用是让 notebook 尽量接近回测，但不强行制造不存在的数据。输出 `filter_capability_df` 用来检查离线模拟缺了哪些真实回测过滤。

In [ ]:
def is_kcb_stock(stock):
    return str(stock).startswith(("688", "689"))


def get_min_lot(stock):
    return KCB_MIN_LOT if is_kcb_stock(stock) else NORMAL_MIN_LOT


def get_price_col(df):
    return first_existing(df.columns, ["last_price", "close", "close_price", "entry_open", "open", "price"])


def build_filter_capability(df):
    checks = [
        ("st_filter", ["is_st", "st", "is_ST"]),
        ("paused_filter", ["paused", "is_paused", "is_suspended"]),
        ("limit_filter", ["high_limit", "low_limit", "last_price", "close", "open"]),
        ("new_stock_filter", ["listing_days", "listed_days", "ipo_days"]),
        ("min_lot_filter", ["last_price", "close", "close_price", "entry_open", "open", "price"]),
    ]
    rows = []
    for name, candidates in checks:
        present = [c for c in candidates if c in df.columns]
        rows.append({
            "filter_name": name,
            "available": bool(present),
            "matched_columns": ",".join(present),
            "note": "used when available" if present else "not available in CSV; JoinQuant backtest remains final judge",
        })
    return pd.DataFrame(rows)


def apply_offline_tradability_filter(month_df):
    df = month_df.copy()
    before = len(df)
    notes = []
    if not USE_OFFLINE_TRADABILITY_FILTER:
        return df, {"before": before, "after": len(df), "removed": 0, "notes": "disabled"}

    for col in ["is_st", "st", "is_ST"]:
        if col in df.columns:
            df = df[~df[col].fillna(False).astype(bool)].copy()
            notes.append("st:" + col)
            break

    for col in ["paused", "is_paused", "is_suspended"]:
        if col in df.columns:
            df = df[~df[col].fillna(False).astype(bool)].copy()
            notes.append("paused:" + col)
            break

    list_col = first_existing(df.columns, ["listing_days", "listed_days", "ipo_days"])
    if list_col is not None:
        df = df[pd.to_numeric(df[list_col], errors="coerce").fillna(99999) >= MIN_LISTING_DAYS].copy()
        notes.append("listing_days:" + list_col)

    price_col = get_price_col(df)
    if price_col is not None:
        price = pd.to_numeric(df[price_col], errors="coerce")
        target_value = INITIAL_CASH_FOR_MIN_LOT / max(1, STOCK_NUM)
        min_values = df[STOCK_COL].map(lambda x: get_min_lot(x)).astype(float) * price
        df = df[(price > 0) & (target_value * 0.98 >= min_values)].copy()
        notes.append("min_lot:" + price_col)

    # 如果同时有涨跌停和价格字段，离线剔除无法新买入的涨跌停；持仓保留无法离线完整模拟，所以这里只是近似。
    price_col = get_price_col(df)
    if price_col is not None and "high_limit" in df.columns and "low_limit" in df.columns:
        price = pd.to_numeric(df[price_col], errors="coerce")
        high_limit = pd.to_numeric(df["high_limit"], errors="coerce")
        low_limit = pd.to_numeric(df["low_limit"], errors="coerce")
        df = df[(price < high_limit) & (price > low_limit)].copy()
        notes.append("limit:" + price_col)

    return df, {"before": before, "after": len(df), "removed": before - len(df), "notes": ",".join(notes) if notes else "no available offline filter columns"}


filter_capability_df = build_filter_capability(df_all)
display(filter_capability_df)


## 模型训练与 pkl 导出

保持 V46/V56 的 bundle schema：`objective=v210_refit_fixed_iter_overlay`，`overlay_mode=direct`，回测文件可以直接加载。这里仍然保留 diag RankIC，但它只是诊断，不参与 early stopping。

In [ ]:
def train_direct_lgb(train_df, feature_cols):
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED
    X_train, y_train, fill_values, train_index = prepare_xy(train_df, feature_cols, TARGET_COL)
    if len(X_train) == 0:
        raise ValueError("empty training matrix")
    model = lgb.train(
        params,
        lgb.Dataset(X_train, label=y_train),
        num_boost_round=max(1, int(FIXED_ITER)),
    )
    pred = np.asarray(model.predict(X_train[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    train_rank_ic = safe_rank_ic(y_train, pred)
    return {
        "model": model,
        "fill_values": fill_values,
        "train_rows": int(len(X_train)),
        "train_rank_ic": train_rank_ic,
    }


def score_with_model(df, model, feature_cols, fill_values):
    X = df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    X = X.fillna(fill_values).fillna(0)
    return np.asarray(model.predict(X[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)


def make_model_id(spec, feature_variant):
    spec = normalize_train_spec(spec)
    train_start_tag = spec["train_start"].strftime("%Y%m%d")
    train_end_tag = spec["train_end"].strftime("%Y%m%d")
    return "v60_{}_{}_start{}_cutoff{}_fixed{}".format(
        feature_variant,
        spec["tag"],
        train_start_tag,
        train_end_tag,
        int(FIXED_ITER),
    )


def needs_v4_adapter(feature_cols):
    adapter_cols = set(HYBRID_LIGHT_EXTRA_COLS)
    return any(c in adapter_cols for c in feature_cols)


def export_bundle(model_id, spec, feature_variant, trained, feature_cols, removed_cols, diag_rank_ic, train_df):
    model_file = "model_candidate_{}.pkl".format(model_id)
    out_path = os.path.join(OUT_DIR, model_file)
    bundle = {
        "objective": "v210_refit_fixed_iter_overlay",
        "research_version": model_id,
        "benchmark": BENCHMARK,
        "train_start": spec["train_start"],
        "train_end": spec["train_end"],
        "test_start": spec["test_start"],
        "label_end": spec["train_end"],
        "boundary_policy": BOUNDARY_POLICY,
        "require_label_end_within_train": bool(not USE_LEGACY_UNSEALED_BOUNDARY),
        "legacy_unsealed_boundary": bool(USE_LEGACY_UNSEALED_BOUNDARY),
        "model_family": MODEL_FAMILY,
        "feature_variant": feature_variant,
        "target_col": TARGET_COL,
        "target_note": "V60 offline portfolio simulator with JQ realized return rebuild: direct LGB alpha_1m, fixed iteration, no early stopping",
        "data_file": DATA_PATH,
        "protocol": "v60_v46_offline_portfolio_realized_rebuild_fixed_cutoff",
        "training_policy": "fixed_window_legacy_unsealed",
        "param_set": "v46_base_ff10_original",
        "base_params": dict(BASE_PARAMS_FF10),
        "base_model": trained["model"],
        "base_feature_cols": list(feature_cols),
        "base_fill_values": dict(trained["fill_values"]),
        "base_best_iter": int(FIXED_ITER),
        "model_iter": int(FIXED_ITER),
        "fixed_iter": int(FIXED_ITER),
        "base_inner_metrics": {
            "train_rank_ic": float(trained["train_rank_ic"]) if not pd.isnull(trained["train_rank_ic"]) else np.nan,
            "diag_rank_ic": float(diag_rank_ic) if not pd.isnull(diag_rank_ic) else np.nan,
        },
        "base_removed_features": list(removed_cols),
        "residual_model": None,
        "residual_feature_cols": [],
        "residual_fill_values": {},
        "overlay_weight": 0.0,
        "overlay_mode": "direct",
        "top_n_candidates": TOP_N_CANDIDATES,
        "stock_num": STOCK_NUM,
        "industry_cap_ratio": INDUSTRY_CAP_RATIO,
        "requires_v4_feature_adapter": bool(needs_v4_adapter(feature_cols)),
        "requires_industry_relative_adapter": False,
        "uses_time_weight": False,
        "uses_sample_weight": False,
        "uses_current_valid_for_training": False,
        "final_role": "v60_realized_return_rebuild_candidate",
        "train_row_count": int(len(train_df)),
        "train_month_count": int(train_df[DATE_COL].nunique()),
        "max_train_rebalance_date": str(train_df[DATE_COL].max().date()),
        "max_train_next_date": str(train_df["next_date"].max().date()),
    }
    if EXPORT_MODELS:
        with open(out_path, "wb") as f:
            pickle.dump(bundle, f, protocol=2)
    return out_path, model_file


def train_one_variant_spec(df_all, variant, spec):
    spec = normalize_train_spec(spec)
    train_df = make_train_df(df_all, spec)
    if train_df.empty:
        raise ValueError("empty train_df for " + str(spec["tag"]))
    diag_fit_df, diag_valid_df = split_diag_valid(train_df)
    feature_cols, removed_cols = select_features_train_only(diag_fit_df, variant["candidate_cols"])
    trained = train_direct_lgb(train_df, feature_cols)
    X_valid, y_valid, _, _ = prepare_xy(diag_valid_df, feature_cols, TARGET_COL, trained["fill_values"])
    valid_pred = np.asarray(trained["model"].predict(X_valid[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    diag_rank_ic = safe_rank_ic(y_valid, valid_pred)
    feature_variant = variant["feature_variant"]
    model_id = make_model_id(spec, feature_variant)
    model_path, model_file = export_bundle(model_id, spec, feature_variant, trained, feature_cols, removed_cols, diag_rank_ic, train_df)
    row = {
        "model_id": model_id,
        "model_family": MODEL_FAMILY,
        "feature_variant": feature_variant,
        "tag": spec["tag"],
        "train_start": spec["train_start"],
        "train_end": spec["train_end"],
        "test_start": spec["test_start"],
        "train_rows": int(len(train_df)),
        "train_months": int(train_df[DATE_COL].nunique()),
        "max_train_next_date": str(train_df["next_date"].max().date()),
        "candidate_feature_count": len(variant["candidate_cols"]),
        "feature_count": len(feature_cols),
        "removed_feature_count": len(removed_cols),
        "train_rank_ic": trained["train_rank_ic"],
        "diag_rank_ic": diag_rank_ic,
        "model_file": model_file,
        "model_path": model_path,
        "feature_cols": ",".join(feature_cols),
        "removed_features": ",".join(removed_cols),
    }
    return row, trained, feature_cols


## 组合模拟逻辑

这一段是 V59 的核心：按照线上使用方式构造组合，而不是只看模型 score 或 label 排名。

流程：

1. 测试月全样本打分。
2. 可选离线可交易过滤。
3. score top30 作为候选池。
4. 候选池内 industry-neutral top6。
5. 等权持有到下一个月。
6. 根据 target 换手估算交易成本，输出 gross/net/excess。

In [ ]:
def build_industry_neutral_targets(sorted_stocks, industry_map, target_num, max_per_industry):
    known_industries = set([
        industry_map.get(stock, "UNKNOWN")
        for stock in sorted_stocks
        if industry_map.get(stock, "UNKNOWN") != "UNKNOWN"
    ])
    if len(known_industries) < 3:
        return sorted_stocks[:min(target_num, len(sorted_stocks))]

    selected = []
    industry_count = {}
    for stock in sorted_stocks:
        industry = industry_map.get(stock, "UNKNOWN")
        if industry == "UNKNOWN":
            continue
        if industry_count.get(industry, 0) == 0:
            selected.append(stock)
            industry_count[industry] = 1
            if len(selected) >= target_num:
                return selected

    for stock in sorted_stocks:
        if stock in selected:
            continue
        industry = industry_map.get(stock, "UNKNOWN")
        if industry == "UNKNOWN":
            continue
        cnt = industry_count.get(industry, 0)
        if cnt < max_per_industry:
            selected.append(stock)
            industry_count[industry] = cnt + 1
            if len(selected) >= target_num:
                return selected

    for stock in sorted_stocks:
        if stock not in selected:
            selected.append(stock)
            if len(selected) >= target_num:
                break
    return selected


def select_model_targets(month_df):
    sorted_all = list(month_df.sort_values("score", ascending=False)[STOCK_COL])
    candidate_stocks = sorted_all[:min(TOP_N_CANDIDATES, len(sorted_all))]
    candidate_df = month_df[month_df[STOCK_COL].isin(candidate_stocks)].copy()
    candidate_sorted = list(candidate_df.sort_values("score", ascending=False)[STOCK_COL])
    industry_map = dict(zip(candidate_df[STOCK_COL], candidate_df[INDUSTRY_COL].fillna("UNKNOWN").astype(str)))
    targets = build_industry_neutral_targets(
        candidate_sorted,
        industry_map,
        STOCK_NUM,
        max(1, int(np.floor(STOCK_NUM * INDUSTRY_CAP_RATIO))),
    )
    return targets, candidate_stocks


def calc_portfolio_return_from_map(ret_map, stocks):
    vals = []
    for stock in stocks:
        v = ret_map.get(stock, np.nan)
        if not pd.isnull(v):
            vals.append(float(v))
    return float(np.mean(vals)) if vals else np.nan


def calc_equal_weight_turnover(prev_targets, targets):
    if len(targets) == 0:
        return 0.0, 0.0, 0
    if prev_targets is None:
        return 1.0, 0.0, 0
    prev = set(prev_targets)
    cur = set(targets)
    overlap = len(prev.intersection(cur))
    one_way = 1.0 - float(overlap) / float(max(1, len(targets)))
    return one_way, one_way, overlap


def calc_trade_cost(prev_targets, targets):
    buy_turnover, sell_turnover, overlap = calc_equal_weight_turnover(prev_targets, targets)
    cost = buy_turnover * BUY_COST_RATE + sell_turnover * SELL_COST_RATE
    if prev_targets is None:
        cost = buy_turnover * BUY_COST_RATE
    return float(cost), float(buy_turnover), float(sell_turnover), int(overlap)


def get_month_benchmark_return(month_df):
    if BENCH_RET_COL is None:
        return np.nan
    s = pd.to_numeric(month_df[BENCH_RET_COL], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return np.nan
    return float(s.iloc[0])


def get_return_map(month_df, col):
    if col is None:
        return {}
    return dict(zip(month_df[STOCK_COL], pd.to_numeric(month_df[col], errors="coerce")))


def board_type(stock):
    s = str(stock)
    if s.startswith("30"):
        return "chinext"
    if s.startswith(("688", "689")):
        return "star"
    return "main"


def summarize_target_board(targets):
    out = {"board_main": 0, "board_chinext": 0, "board_star": 0}
    for stock in targets:
        b = board_type(stock)
        out["board_" + b] = out.get("board_" + b, 0) + 1
    return out


def evaluate_trained_model_portfolio(df_all, meta_row, trained, feature_cols):
    spec = {
        "tag": meta_row["tag"],
        "train_start": meta_row["train_start"],
        "train_end": meta_row["train_end"],
        "test_start": meta_row["test_start"],
    }
    test_df = make_test_df(df_all, spec)
    if test_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    test_df = test_df.copy()
    test_df["score"] = score_with_model(test_df, trained["model"], feature_cols, trained["fill_values"])
    test_df["score_rank_pct"] = test_df.groupby(DATE_COL)["score"].rank(pct=True)
    test_df["realized_rank_pct"] = test_df.groupby(DATE_COL)[TARGET_COL].rank(pct=True)

    monthly_rows = []
    filter_rows = []
    panel_parts = []
    prev_targets = None
    groups = list(test_df.groupby(DATE_COL))
    for rebalance_date, raw_month_df in progress_iter(groups, total=len(groups), desc="portfolio eval %s %s" % (meta_row["feature_variant"], meta_row["tag"])):
        raw_month_df = raw_month_df.dropna(subset=[TARGET_COL, "score"]).copy()
        if raw_month_df.empty:
            continue
        month_df, filter_info = apply_offline_tradability_filter(raw_month_df)
        month_df = month_df.dropna(subset=[TARGET_COL, "score"]).copy()
        filter_info.update({
            "model_id": meta_row["model_id"],
            "feature_variant": meta_row["feature_variant"],
            "tag": meta_row["tag"],
            "rebalance_date": rebalance_date,
        })
        filter_rows.append(filter_info)
        if month_df.empty:
            continue

        targets, candidate_top30 = select_model_targets(month_df)
        cost, buy_turnover, sell_turnover, overlap = calc_trade_cost(prev_targets, targets)
        raw_ret_map = get_return_map(month_df, RAW_RET_COL)
        alpha_ret_map = get_return_map(month_df, ALPHA_RET_COL)
        gross_raw_ret = calc_portfolio_return_from_map(raw_ret_map, targets) if RAW_RET_COL is not None else np.nan
        gross_alpha_ret = calc_portfolio_return_from_map(alpha_ret_map, targets) if ALPHA_RET_COL is not None else np.nan
        benchmark_ret = get_month_benchmark_return(month_df)

        if pd.isnull(gross_raw_ret):
            # 如果没有 raw return，只能用 alpha 近似净超额。
            net_raw_ret = np.nan
            net_excess_ret = gross_alpha_ret - cost if not pd.isnull(gross_alpha_ret) else np.nan
        else:
            net_raw_ret = (1.0 + gross_raw_ret) * (1.0 - cost) - 1.0
            if not pd.isnull(benchmark_ret):
                net_excess_ret = net_raw_ret - benchmark_ret
            elif not pd.isnull(gross_alpha_ret):
                net_excess_ret = gross_alpha_ret - cost
            else:
                net_excess_ret = np.nan

        actual_top6 = list(month_df.sort_values(TARGET_COL, ascending=False)[STOCK_COL].head(REALIZED_TOP_N))
        actual_top20 = set(list(month_df.sort_values(TARGET_COL, ascending=False)[STOCK_COL].head(REALIZED_TOP20_N)))
        actual_top6_alpha = calc_portfolio_return_from_map(alpha_ret_map, actual_top6)
        selected_rows = month_df.set_index(STOCK_COL).reindex(targets)
        hit_top6 = len(set(targets).intersection(set(actual_top6)))
        board = summarize_target_board(targets)

        row = {
            "model_id": meta_row["model_id"],
            "feature_variant": meta_row["feature_variant"],
            "tag": meta_row["tag"],
            "train_start": meta_row["train_start"],
            "train_end": meta_row["train_end"],
            "test_start": meta_row["test_start"],
            "rebalance_date": rebalance_date,
            "next_date": month_df["next_date"].iloc[0] if "next_date" in month_df.columns else pd.NaT,
            "target_count": len(targets),
            "candidate_count": len(candidate_top30),
            "gross_raw_ret": gross_raw_ret,
            "gross_alpha_ret": gross_alpha_ret,
            "benchmark_ret": benchmark_ret,
            "trade_cost": cost,
            "net_raw_ret": net_raw_ret,
            "net_excess_ret": net_excess_ret,
            "buy_turnover": buy_turnover,
            "sell_turnover": sell_turnover,
            "target_overlap_prev": overlap,
            "rank_ic": safe_rank_ic(month_df["score"], month_df[TARGET_COL]),
            "actual_top6_alpha": actual_top6_alpha,
            "oracle_gap_alpha": actual_top6_alpha - gross_alpha_ret if not pd.isnull(gross_alpha_ret) else np.nan,
            "target_avg_realized_rank": float(selected_rows["realized_rank_pct"].mean()),
            "target_worst_realized_rank": float(selected_rows["realized_rank_pct"].min()),
            "hit_top6": int(hit_top6),
            "fn_top6_count": int(max(0, REALIZED_TOP_N - hit_top6)),
            "fp_top6_count": int(max(0, len(targets) - hit_top6)),
            "fp_not_top20_count": int(len([s for s in targets if s not in actual_top20])),
            "targets": ",".join(targets),
            "candidate_top30": ",".join(candidate_top30),
            "actual_top6": ",".join(actual_top6),
        }
        row.update(board)
        monthly_rows.append(row)

        month_df["model_id"] = meta_row["model_id"]
        month_df["feature_variant"] = meta_row["feature_variant"]
        month_df["tag"] = meta_row["tag"]
        panel_parts.append(month_df)
        prev_targets = list(targets)

    monthly_df = pd.DataFrame(monthly_rows)
    if not monthly_df.empty:
        monthly_df = monthly_df.sort_values(["model_id", "rebalance_date"]).reset_index(drop=True)
        monthly_df["net_nav"] = monthly_df.groupby("model_id")["net_raw_ret"].transform(lambda s: calc_nav(s).values if len(s) else s)
        monthly_df["net_excess_nav"] = monthly_df.groupby("model_id")["net_excess_ret"].transform(lambda s: calc_nav(s).values if len(s) else s)
    score_panel_df = pd.concat(panel_parts, ignore_index=True) if panel_parts else pd.DataFrame()
    filter_df = pd.DataFrame(filter_rows)
    return monthly_df, score_panel_df, filter_df


## 同约束随机基线

这里的随机不是随机 6 只股票，而是在同一测试月、同一离线过滤后股票池、同样 industry-neutral top6 约束下随机抽取。这个指标比平均 rank 更贴近真实应用场景。

In [ ]:
def encode_industries(industry_values):
    codes = []
    mapping = {}
    next_code = 0
    for x in industry_values:
        key = str(x) if not pd.isnull(x) else "UNKNOWN"
        if key == "UNKNOWN":
            codes.append(-1)
            continue
        if key not in mapping:
            mapping[key] = next_code
            next_code += 1
        codes.append(mapping[key])
    return np.asarray(codes, dtype=np.int32), len(mapping)


def select_random_indices_industry_neutral(perm, industry_codes, target_num, max_per_industry, known_industry_count):
    if len(perm) <= target_num or known_industry_count < 3:
        return perm[:min(target_num, len(perm))]

    selected = []
    selected_flag = np.zeros(len(industry_codes), dtype=np.bool_)
    industry_count = {}

    for idx in perm:
        ind = int(industry_codes[idx])
        if ind < 0:
            continue
        if industry_count.get(ind, 0) == 0:
            selected.append(idx)
            selected_flag[idx] = True
            industry_count[ind] = 1
            if len(selected) >= target_num:
                return np.asarray(selected, dtype=np.int32)

    for idx in perm:
        if selected_flag[idx]:
            continue
        ind = int(industry_codes[idx])
        if ind < 0:
            continue
        cnt = industry_count.get(ind, 0)
        if cnt < max_per_industry:
            selected.append(idx)
            selected_flag[idx] = True
            industry_count[ind] = cnt + 1
            if len(selected) >= target_num:
                return np.asarray(selected, dtype=np.int32)

    for idx in perm:
        if not selected_flag[idx]:
            selected.append(idx)
            if len(selected) >= target_num:
                break
    return np.asarray(selected, dtype=np.int32)


def stable_text_seed(text):
    text = str(text)
    total = 0
    for i, ch in enumerate(text):
        total += (i + 1) * ord(ch)
    return int(total % 100000)


def random_distribution_for_month_fast(month_df, return_col, sim_n, seed_key):
    if return_col is None:
        return np.asarray([], dtype=float)
    rets = pd.to_numeric(month_df[return_col], errors="coerce").replace([np.inf, -np.inf], np.nan).values.astype(float)
    valid = np.isfinite(rets)
    rets = rets[valid]
    industries = month_df.loc[valid, INDUSTRY_COL].fillna("UNKNOWN").astype(str).values
    n = len(rets)
    if n == 0:
        return np.asarray([], dtype=float)
    industry_codes, known_industry_count = encode_industries(industries)
    max_per_industry = max(1, int(np.floor(STOCK_NUM * INDUSTRY_CAP_RATIO)))
    rng = np.random.RandomState(seed_key)
    out = np.empty(int(sim_n), dtype=float)
    base_idx = np.arange(n, dtype=np.int32)
    for i in range(int(sim_n)):
        perm = rng.permutation(base_idx)
        picked = select_random_indices_industry_neutral(
            perm,
            industry_codes,
            STOCK_NUM,
            max_per_industry,
            known_industry_count,
        )
        out[i] = float(np.nanmean(rets[picked])) if len(picked) else np.nan
    return out


def add_random_baseline(monthly_df, score_panel_df):
    rows = []
    if monthly_df.empty:
        return pd.DataFrame()
    iterator = progress_iter(monthly_df.iterrows(), total=len(monthly_df), desc="random baseline")
    for _, row in iterator:
        model_id = str(row["model_id"])
        dt = pd.Timestamp(row["rebalance_date"])
        month_df = score_panel_df[(score_panel_df["model_id"].astype(str) == model_id) & (score_panel_df[DATE_COL] == dt)].copy()
        month_df = month_df.dropna(subset=[TARGET_COL, "score"])
        if month_df.empty:
            continue
        seed_key = int(pd.Timestamp(dt).strftime("%Y%m%d")) + stable_text_seed(model_id)
        rand_alpha = random_distribution_for_month_fast(month_df, ALPHA_RET_COL, RANDOM_SIM_N, seed_key)
        rand_raw = random_distribution_for_month_fast(month_df, RAW_RET_COL, RANDOM_SIM_N, seed_key + 17)
        out = dict(row)
        if len(rand_alpha):
            rand_alpha = rand_alpha[np.isfinite(rand_alpha)]
            if len(rand_alpha) and not pd.isnull(row.get("gross_alpha_ret", np.nan)):
                out.update({
                    "random_alpha_percentile": float((rand_alpha <= float(row["gross_alpha_ret"])).mean()),
                    "random_alpha_mean": float(np.nanmean(rand_alpha)),
                    "random_alpha_median": float(np.nanmedian(rand_alpha)),
                    "random_alpha_p10": float(np.nanpercentile(rand_alpha, 10)),
                    "random_alpha_p90": float(np.nanpercentile(rand_alpha, 90)),
                })
        if len(rand_raw):
            rand_raw = rand_raw[np.isfinite(rand_raw)]
            if len(rand_raw) and not pd.isnull(row.get("gross_raw_ret", np.nan)):
                out.update({
                    "random_raw_percentile": float((rand_raw <= float(row["gross_raw_ret"])).mean()),
                    "random_raw_mean": float(np.nanmean(rand_raw)),
                    "random_raw_median": float(np.nanmedian(rand_raw)),
                    "random_raw_p10": float(np.nanpercentile(rand_raw, 10)),
                    "random_raw_p90": float(np.nanpercentile(rand_raw, 90)),
                })
        rows.append(out)
    return pd.DataFrame(rows)


## 批量训练、离线组合回测与导出

每个 job 都会训练模型、导出 pkl、生成月度组合和 score panel。进度条用于避免长循环时看不到状态。

In [ ]:
manifest_rows = []
monthly_parts = []
score_panel_parts = []
filter_parts = []
trained_map = {}
feature_cols_map = {}

jobs = []
for variant in FEATURE_VARIANTS:
    for spec in TRAIN_WINDOW_SPECS:
        jobs.append((variant, spec))

for variant, spec in progress_iter(jobs, total=len(jobs), desc="train/eval variants"):
    spec_norm = normalize_train_spec(spec)
    print("\ntrain", variant["feature_variant"], spec_norm["tag"], spec_norm["train_start"].date(), spec_norm["train_end"].date())
    meta_row, trained, feature_cols = train_one_variant_spec(df_all, variant, spec_norm)
    manifest_rows.append(meta_row)
    trained_map[meta_row["model_id"]] = trained
    feature_cols_map[meta_row["model_id"]] = list(feature_cols)
    monthly_df_one, score_panel_df_one, filter_df_one = evaluate_trained_model_portfolio(df_all, meta_row, trained, feature_cols)
    if not monthly_df_one.empty:
        monthly_parts.append(monthly_df_one)
    if not score_panel_df_one.empty:
        score_panel_parts.append(score_panel_df_one)
    if not filter_df_one.empty:
        filter_parts.append(filter_df_one)
    gc.collect()

manifest_df = pd.DataFrame(manifest_rows)
monthly_df = pd.concat(monthly_parts, ignore_index=True) if monthly_parts else pd.DataFrame()
score_panel_df = pd.concat(score_panel_parts, ignore_index=True) if score_panel_parts else pd.DataFrame()
filter_report_df = pd.concat(filter_parts, ignore_index=True) if filter_parts else pd.DataFrame()

print("manifest:", manifest_df.shape)
print("monthly portfolio:", monthly_df.shape)
print("score panel:", score_panel_df.shape)
print("filter report:", filter_report_df.shape)
display(manifest_df[["feature_variant", "tag", "train_start", "train_end", "feature_count", "train_rank_ic", "diag_rank_ic", "model_file"]])
display(monthly_df.head())


## JoinQuant 真实收益重算

V59 的主要问题是只用 `alpha_1m` proxy 评价组合，和 JoinQuant 回测的 close-to-close realized return 仍有差异。

这一段只对每月已经选出的 target 股票拉日线 close，按回测 monitor 的同口径计算：

- `start_date = rebalance_date`
- `end_date = next_date`
- `fq='pre'`
- `skip_paused=False`
- `fill_paused=True`
- 等权 close-to-close return
- 同期 benchmark close-to-close return

如果当前环境没有 JoinQuant `get_price`，`RUN_JQ_REALIZED_REBUILD='AUTO'` 会跳过，不影响 CSV proxy 部分跑完。

In [ ]:
def parse_target_list(x):
    if pd.isnull(x):
        return []
    return [s.strip() for s in str(x).split(",") if s.strip()]


def jq_get_price_safe(securities, start_date, end_date, fields):
    try:
        try:
            return get_price(
                securities,
                start_date=start_date,
                end_date=end_date,
                frequency="daily",
                fields=fields,
                skip_paused=False,
                fq=REALIZED_PRICE_FQ,
                panel=False,
                fill_paused=True,
            )
        except TypeError:
            return get_price(
                securities,
                start_date=start_date,
                end_date=end_date,
                frequency="daily",
                fields=fields,
                skip_paused=False,
                fq=REALIZED_PRICE_FQ,
                panel=False,
            )
    except NameError:
        raise RuntimeError("JoinQuant get_price is not available in this environment")


def calc_jq_equal_weight_close_monitor(stocks, start_date, end_date):
    result = {
        "realized_raw_ret": np.nan,
        "realized_mdd": np.nan,
        "realized_win_rate": np.nan,
        "realized_valid_count": 0,
    }
    stocks = unique_keep_order(list(stocks))
    if len(stocks) == 0 or pd.isnull(start_date) or pd.isnull(end_date):
        return result
    try:
        price_df = jq_get_price_safe(
            stocks,
            pd.Timestamp(start_date).strftime("%Y-%m-%d"),
            pd.Timestamp(end_date).strftime("%Y-%m-%d"),
            [REALIZED_PRICE_FIELD],
        )
    except RuntimeError:
        raise
    except Exception as err:
        result["realized_error"] = str(err)
        return result
    if price_df is None or price_df.empty:
        return result
    if "time" not in price_df.columns or "code" not in price_df.columns or REALIZED_PRICE_FIELD not in price_df.columns:
        return result
    price_df = price_df.copy()
    price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
    close_mat = price_df.pivot_table(index="time", columns="code", values=REALIZED_PRICE_FIELD).sort_index()
    if close_mat.empty or len(close_mat) < 2:
        return result
    first = close_mat.iloc[0].replace(0, np.nan)
    last = close_mat.iloc[-1]
    stock_rets = (last / first - 1.0).replace([np.inf, -np.inf], np.nan).dropna()
    if len(stock_rets) == 0:
        return result
    norm = close_mat.reindex(columns=stock_rets.index) / first.reindex(stock_rets.index)
    curve = norm.mean(axis=1).dropna()
    if len(curve) > 0:
        drawdown = curve / curve.cummax() - 1.0
        result["realized_mdd"] = float(drawdown.min())
    result["realized_raw_ret"] = float(stock_rets.mean())
    result["realized_win_rate"] = float((stock_rets > 0).mean())
    result["realized_valid_count"] = int(len(stock_rets))
    return result


def calc_jq_benchmark_close_return(benchmark, start_date, end_date):
    if benchmark is None or pd.isnull(start_date) or pd.isnull(end_date):
        return np.nan
    try:
        bench_df = jq_get_price_safe(
            benchmark,
            pd.Timestamp(start_date).strftime("%Y-%m-%d"),
            pd.Timestamp(end_date).strftime("%Y-%m-%d"),
            [REALIZED_PRICE_FIELD],
        )
    except RuntimeError:
        raise
    except Exception:
        return np.nan
    if bench_df is None or bench_df.empty or REALIZED_PRICE_FIELD not in bench_df.columns:
        return np.nan
    close = pd.to_numeric(bench_df[REALIZED_PRICE_FIELD], errors="coerce").dropna()
    if len(close) < 2 or close.iloc[0] == 0:
        return np.nan
    return float(close.iloc[-1] / close.iloc[0] - 1.0)


def rebuild_realized_returns_from_jq(monthly_df):
    if monthly_df.empty:
        return monthly_df.copy(), pd.DataFrame()
    out_rows = []
    status_rows = []
    iterator = progress_iter(monthly_df.iterrows(), total=len(monthly_df), desc="jq realized rebuild")
    for _, row in iterator:
        out = dict(row)
        stocks = parse_target_list(row.get("targets", ""))
        start_date = row.get("rebalance_date")
        end_date = row.get("next_date")
        status = {
            "model_id": row.get("model_id"),
            "feature_variant": row.get("feature_variant"),
            "tag": row.get("tag"),
            "rebalance_date": start_date,
            "next_date": end_date,
            "target_count": len(stocks),
            "realized_rebuild_ok": False,
            "realized_rebuild_note": "",
        }
        try:
            stats = calc_jq_equal_weight_close_monitor(stocks, start_date, end_date)
            bench_ret = calc_jq_benchmark_close_return(BENCHMARK, start_date, end_date)
            realized_raw = stats.get("realized_raw_ret", np.nan)
            trade_cost = float(row.get("trade_cost", 0.0)) if not pd.isnull(row.get("trade_cost", np.nan)) else 0.0
            realized_net_raw = (1.0 + realized_raw) * (1.0 - trade_cost) - 1.0 if not pd.isnull(realized_raw) else np.nan
            realized_excess = realized_raw - bench_ret if not pd.isnull(realized_raw) and not pd.isnull(bench_ret) else np.nan
            realized_net_excess = realized_net_raw - bench_ret if not pd.isnull(realized_net_raw) and not pd.isnull(bench_ret) else np.nan
            out.update(stats)
            out.update({
                "realized_benchmark_ret": bench_ret,
                "realized_excess_ret": realized_excess,
                "realized_net_raw_ret": realized_net_raw,
                "realized_net_excess_ret": realized_net_excess,
            })
            status["realized_rebuild_ok"] = not pd.isnull(realized_raw)
            status["realized_rebuild_note"] = stats.get("realized_error", "") if "realized_error" in stats else "ok"
        except RuntimeError as err:
            raise
        except Exception as err:
            status["realized_rebuild_note"] = str(err)
        out_rows.append(out)
        status_rows.append(status)
    out_df = pd.DataFrame(out_rows)
    status_df = pd.DataFrame(status_rows)
    if "realized_net_raw_ret" in out_df.columns:
        out_df = out_df.sort_values(["model_id", "rebalance_date"]).reset_index(drop=True)
        out_df["realized_net_nav"] = out_df.groupby("model_id")["realized_net_raw_ret"].transform(lambda s: calc_nav(s).values if len(s) else s)
        out_df["realized_net_excess_nav"] = out_df.groupby("model_id")["realized_net_excess_ret"].transform(lambda s: calc_nav(s).values if len(s) else s)
    return out_df, status_df


def maybe_rebuild_realized_returns(monthly_df):
    mode = RUN_JQ_REALIZED_REBUILD
    if mode is False or str(mode).upper() == "FALSE":
        print("JQ realized rebuild disabled")
        out = monthly_df.copy()
        out["realized_rebuild_available"] = False
        return out, pd.DataFrame()
    try:
        out, status = rebuild_realized_returns_from_jq(monthly_df)
        out["realized_rebuild_available"] = True
        print("JQ realized rebuild done:", out.shape)
        return out, status
    except RuntimeError as err:
        if str(mode).upper() == "AUTO":
            print("JQ realized rebuild skipped:", err)
            out = monthly_df.copy()
            out["realized_rebuild_available"] = False
            return out, pd.DataFrame([{"realized_rebuild_ok": False, "realized_rebuild_note": str(err)}])
        raise


realized_monthly_df, realized_rebuild_status_df = maybe_rebuild_realized_returns(monthly_df)
analysis_monthly_df = realized_monthly_df.copy()
print("analysis monthly:", analysis_monthly_df.shape)
if "realized_net_excess_ret" in analysis_monthly_df.columns:
    display(analysis_monthly_df[[
        "feature_variant", "tag", "rebalance_date", "next_date", "realized_raw_ret",
        "realized_benchmark_ret", "realized_net_excess_ret", "targets"
    ]].head())


## 汇总与验收排序

优先看 `net_raw_cum_ret / net_excess_cum_ret / net_raw_max_drawdown / turnover / random percentile`。如果这里的排序和 JoinQuant 回测经常相反，说明离线模拟仍然缺关键字段或逻辑。

In [ ]:
def summarize_portfolio_monthly(eval_monthly_df):
    rows = []
    if eval_monthly_df.empty:
        return pd.DataFrame()
    group_cols = ["feature_variant", "tag", "model_id", "train_start", "train_end", "test_start"]
    for keys, gdf in eval_monthly_df.groupby(group_cols):
        gdf = gdf.sort_values("rebalance_date").copy()
        proxy_raw = summarize_return_series(gdf["net_raw_ret"]) if "net_raw_ret" in gdf.columns else {}
        proxy_excess = summarize_return_series(gdf["net_excess_ret"]) if "net_excess_ret" in gdf.columns else {}
        alpha_proxy = summarize_return_series(gdf["gross_alpha_ret"]) if "gross_alpha_ret" in gdf.columns else {}
        realized_raw = summarize_return_series(gdf["realized_net_raw_ret"]) if "realized_net_raw_ret" in gdf.columns else {}
        realized_excess = summarize_return_series(gdf["realized_net_excess_ret"]) if "realized_net_excess_ret" in gdf.columns else {}
        ric = gdf["rank_ic"].replace([np.inf, -np.inf], np.nan).dropna()
        realized_col = "realized_net_raw_ret" if "realized_net_raw_ret" in gdf.columns and gdf["realized_net_raw_ret"].notnull().any() else "net_raw_ret"
        drop_top1 = np.nan
        if realized_col in gdf.columns and gdf[realized_col].notnull().any() and len(gdf) > 1:
            best_idx = gdf[realized_col].idxmax()
            drop_top1 = float((1.0 + gdf.drop(best_idx)[realized_col].fillna(0)).prod() - 1.0)
        rows.append({
            "feature_variant": keys[0],
            "tag": keys[1],
            "model_id": keys[2],
            "train_start": keys[3],
            "train_end": keys[4],
            "test_start": keys[5],
            "months": int(len(gdf)),
            "proxy_alpha_cum_ret": alpha_proxy.get("cum_ret", np.nan),
            "proxy_net_raw_cum_ret": proxy_raw.get("cum_ret", np.nan),
            "proxy_net_excess_cum_ret": proxy_excess.get("cum_ret", np.nan),
            "proxy_net_excess_max_drawdown": proxy_excess.get("max_drawdown", np.nan),
            "realized_net_raw_cum_ret": realized_raw.get("cum_ret", np.nan),
            "realized_net_raw_mean_ret": realized_raw.get("mean_ret", np.nan),
            "realized_net_raw_win_rate": realized_raw.get("win_rate", np.nan),
            "realized_net_raw_max_drawdown": realized_raw.get("max_drawdown", np.nan),
            "realized_net_excess_cum_ret": realized_excess.get("cum_ret", np.nan),
            "realized_net_excess_mean_ret": realized_excess.get("mean_ret", np.nan),
            "realized_net_excess_win_rate": realized_excess.get("win_rate", np.nan),
            "realized_net_excess_max_drawdown": realized_excess.get("max_drawdown", np.nan),
            "avg_trade_cost": float(gdf["trade_cost"].mean()) if "trade_cost" in gdf.columns else np.nan,
            "avg_buy_turnover": float(gdf["buy_turnover"].mean()) if "buy_turnover" in gdf.columns else np.nan,
            "avg_sell_turnover": float(gdf["sell_turnover"].mean()) if "sell_turnover" in gdf.columns else np.nan,
            "avg_board_main": float(gdf["board_main"].mean()) if "board_main" in gdf.columns else np.nan,
            "avg_board_chinext": float(gdf["board_chinext"].mean()) if "board_chinext" in gdf.columns else np.nan,
            "avg_board_star": float(gdf["board_star"].mean()) if "board_star" in gdf.columns else np.nan,
            "avg_hit_top6": float(gdf["hit_top6"].mean()) if "hit_top6" in gdf.columns else np.nan,
            "avg_fp_not_top20_count": float(gdf["fp_not_top20_count"].mean()) if "fp_not_top20_count" in gdf.columns else np.nan,
            "avg_oracle_gap_alpha": float(gdf["oracle_gap_alpha"].mean()) if "oracle_gap_alpha" in gdf.columns else np.nan,
            "avg_target_realized_rank": float(gdf["target_avg_realized_rank"].mean()) if "target_avg_realized_rank" in gdf.columns else np.nan,
            "rank_ic_mean": float(ric.mean()) if len(ric) else np.nan,
            "rank_ic_ir": float(ric.mean() / ric.std()) if len(ric) > 1 and ric.std() > 0 else np.nan,
            "drop_top1_realized_or_proxy_cum_ret": drop_top1,
        })
    out = pd.DataFrame(rows)
    sort_col = "realized_net_raw_cum_ret" if "realized_net_raw_cum_ret" in out.columns and out["realized_net_raw_cum_ret"].notnull().any() else "proxy_net_excess_cum_ret"
    return out.sort_values(["tag", sort_col], ascending=[True, False])


random_monthly_df = add_random_baseline(monthly_df, score_panel_df)
portfolio_summary_df = summarize_portfolio_monthly(analysis_monthly_df)

if not random_monthly_df.empty:
    rand_cols = ["model_id", "random_alpha_percentile", "random_raw_percentile"]
    keep_cols = [c for c in rand_cols if c in random_monthly_df.columns and c != "model_id"]
    if keep_cols:
        rand_agg = random_monthly_df.groupby("model_id")[keep_cols].mean().reset_index()
        portfolio_summary_df = portfolio_summary_df.merge(rand_agg, on="model_id", how="left")

model_summary_df = manifest_df.merge(
    portfolio_summary_df,
    on=["model_id", "feature_variant", "tag", "train_start", "train_end", "test_start"],
    how="left",
)

show_cols = [
    "feature_variant", "tag", "train_start", "train_end", "feature_count",
    "realized_net_raw_cum_ret", "realized_net_excess_cum_ret", "realized_net_raw_max_drawdown",
    "proxy_net_excess_cum_ret", "avg_buy_turnover", "random_alpha_percentile",
    "avg_hit_top6", "avg_fp_not_top20_count", "diag_rank_ic", "model_file",
]
print("model summary")
display(model_summary_df[[c for c in show_cols if c in model_summary_df.columns]].sort_values(["tag", "realized_net_raw_cum_ret" if "realized_net_raw_cum_ret" in model_summary_df.columns else "proxy_net_excess_cum_ret"], ascending=[True, False]))

print("analysis monthly sample")
display(analysis_monthly_df.head(20))


## 最新 target 与导出文件

最终进入 JoinQuant 回测前，先看 `v59_model_summary.csv` 和 `v59_monthly_portfolio.csv`。如果某个模型离线明显胜出，再用对应 pkl 跑真实回测验收。

In [ ]:
def build_latest_targets(eval_monthly_df):
    if eval_monthly_df.empty:
        return pd.DataFrame()
    rows = []
    for model_id, gdf in eval_monthly_df.groupby("model_id"):
        dt = gdf["rebalance_date"].max()
        rows.append(gdf[gdf["rebalance_date"] == dt].iloc[0].to_dict())
    return pd.DataFrame(rows)


latest_targets_df = build_latest_targets(analysis_monthly_df)

variant_manifest_df.to_csv(os.path.join(OUT_DIR, "v60_feature_variant_manifest.csv"), index=False)
filter_capability_df.to_csv(os.path.join(OUT_DIR, "v60_filter_capability.csv"), index=False)
manifest_df.to_csv(os.path.join(OUT_DIR, "v60_model_manifest.csv"), index=False)
model_summary_df.to_csv(os.path.join(OUT_DIR, "v60_model_summary.csv"), index=False)
monthly_df.to_csv(os.path.join(OUT_DIR, "v60_monthly_portfolio_proxy.csv"), index=False)
realized_monthly_df.to_csv(os.path.join(OUT_DIR, "v60_monthly_realized.csv"), index=False)
realized_rebuild_status_df.to_csv(os.path.join(OUT_DIR, "v60_realized_rebuild_status.csv"), index=False)
score_panel_df.to_csv(os.path.join(OUT_DIR, "v60_score_panel.csv"), index=False)
filter_report_df.to_csv(os.path.join(OUT_DIR, "v60_filter_report.csv"), index=False)
random_monthly_df.to_csv(os.path.join(OUT_DIR, "v60_random_monthly.csv"), index=False)
latest_targets_df.to_csv(os.path.join(OUT_DIR, "v60_latest_targets.csv"), index=False)

print("saved outputs to:", OUT_DIR)
print("model pkls exported:", bool(EXPORT_MODELS))
for name in [
    "v60_model_manifest.csv",
    "v60_model_summary.csv",
    "v60_monthly_portfolio_proxy.csv",
    "v60_monthly_realized.csv",
    "v60_random_monthly.csv",
    "v60_latest_targets.csv",
]:
    print(os.path.join(OUT_DIR, name))

show_cols = ["feature_variant", "tag", "rebalance_date", "targets", "realized_raw_ret", "realized_net_excess_ret", "net_excess_ret"]
display(latest_targets_df[[c for c in show_cols if c in latest_targets_df.columns]])


## 结论阅读模板

跑完后按这个顺序看：

1. `v60_filter_capability.csv`：确认离线模拟缺哪些真实回测过滤。缺得越多，notebook 结论越需要回测复核。
2. `v60_model_summary.csv`：先按同一个 `tag` 比较 `realized_net_raw_cum_ret / realized_net_excess_cum_ret / realized drawdown / turnover`。
3. `v60_monthly_portfolio_proxy.csv / v60_monthly_realized.csv`：看 full/light 差异来自哪些月份，不只看总收益。
4. `v60_random_monthly.csv`：看模型 top6 相对同约束随机 top6 的分位，避免被单月 avg rank 误导。
5. `v60_latest_targets.csv`：人工检查最新 target 是否集中在某个板块/行业。
6. 只有 V60 明显过关的 pkl，才进入 `jq_backtest_v46_legacy_unsealed_monitor.py` 回测验收。

接受规则：如果 V60 和 JoinQuant 回测排序持续相反，优先修离线模拟字段和组合规则，不继续基于错误 proxy 优化模型；优先检查 realized rebuild 是否可用。

V60 结论优先级：如果 `realized_rebuild_available=True`，以 realized columns 排序；如果为 False，说明当前环境没有 JoinQuant API，输出仍是 proxy，只能作为诊断。